# 🔍 Deepfake Detection using Swin Transformer (Face-Only Pipeline)
## A Multi-Modal Approach to Deepfake Detection via Behavioral, Spatial, and Temporal Cues

**Model:** `microsoft/swin-base-patch4-window7-224-in22k` (Swin Transformer)  
**Task:** Binary Classification (Real vs Fake)  
**Pipeline:** Video → Face Extraction (MTCNN) → Swin Transformer → Prediction

### Key Research Papers:
1. Liu et al. (2021) - "Swin Transformer: Hierarchical Vision Transformer using Shifted Windows" (ICCV Best Paper)
2. Coccomini et al. (2022) - "Combining EfficientNet and Vision Transformers for Video Deepfake Detection"
3. Rossler et al. (2019) - "FaceForensics++: Learning to Detect Manipulated Facial Images"
4. Li et al. (2020) - "Celeb-DF: A Challenge Dataset for DeepFake Forensics"
5. Dosovitskiy et al. (2020) - "An Image is Worth 16x16 Words: Transformers for Image Recognition"
6. Wodajo & Atnafu (2021) - "Deepfake Video Detection Using CNNs and Vision Transformers"

# 1. Setup & Installation

In [ ]:
from google.colab import drive
import os
import shutil

# Ensure the mount point is clean before attempting to mount
if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    print("Clearing non-mountpoint /content/drive directory...")
    try:
        shutil.rmtree('/content/drive')
    except OSError as e:
        print(f"Error removing directory: {e}. Attempting to create anyway.")
    os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install required packages (uncomment if needed)
!pip install facenet-pytorch transformers datasets evaluate accelerate torchvision scikit-learn pytorchvideo

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import cv2
import torch
import shutil
import numpy as np
from PIL import Image
from pathlib import Path
from collections import Counter

# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


# 2. Configuration

In [ ]:
# Model selection (Swin Transformer - best for deepfake detection)
#MODEL_NAME = "microsoft/swin-base-patch4-window7-224-in22k"
MODEL_NAME = "google/vit-base-patch16-224"

# Alternative models (uncomment to try):
# MODEL_NAME = "google/vit-large-patch16-224"        # ViT-Large (~307M params)
# MODEL_NAME = "microsoft/beit-large-patch16-224-pt22k-ft22k"  # BEiT-Large

# Paths
# VIDEO_ROOT = "/content/drive/MyDrive/FYP/dataset" # Corrected path to dataset root
FACE_OUTPUT = "/content/drive/MyDrive/FYP/dataset/Frames"
# OUTPUT_DIR = "/content/drive/MyDrive/FYP/TrainedModel_Swin"

# Training hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 8
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# Frame extraction settings
SAMPLE_RATE = 4       # Take every Nth frame
MAX_FRAMES = 30       # Max frames per video

# Video limits for data processing
MAX_REAL_VIDEOS = 552
START_REAL_VIDEO_ID = 0 # Start processing real videos from this ID
MAX_FAKE_VIDEOS_PER_CATEGORY = 300 # Limit for each fake sub-category

# Label mapping
LABEL_MAP = {"Real": 0, "Fake": 1}
ID2LABEL = {0: "Real", 1: "Fake"}
LABEL2ID = {"Real": 0, "Fake": 1}

print("Configuration loaded successfully!")
print(f"Model: {MODEL_NAME}")

Configuration loaded successfully!
Model: google/vit-base-patch16-224


# 3. Face Extraction from Videos (MTCNN)

In [ ]:
#!pip install facenet-pytorch

In [ ]:
# from facenet_pytorch import MTCNN

# VIDEO_ROOT = "/content/drive/MyDrive/FYP/dataset"
# FACE_OUTPUT = "/content/drive/MyDrive/FYP/dataset/Frames" # Corrected path
# OUTPUT_DIR = "/content/drive/MyDrive/FYP/TrainedModel_Swin"


# # Initialize MTCNN face detector
# detector = MTCNN(
#     image_size=224,
#     margin=20,
#     keep_all=False,
#     post_process=False,
#     device=device
# )

# def extract_faces_from_video(video_path, face_output_folder,
#                               sample_rate=SAMPLE_RATE, max_frames=MAX_FRAMES):
#     """
#     Extract face crops from a video using MTCNN.
#     Only saves frames where a face is detected.
#     Full-frame/global images are NOT saved (face-only pipeline).
#     """
#     cap = cv2.VideoCapture(video_path)

#     if not cap.isOpened():
#         print(f"Error opening video: {video_path}")
#         return 0

#     os.makedirs(face_output_folder, exist_ok=True)

#     frame_id = 0   # saved face count
#     count = 0      # video frame counter

#     while True:
#         ret, frame = cap.read()
#         if not ret or frame_id >= max_frames:
#             break

#         if count % sample_rate == 0:
#             # Convert BGR → RGB for MTCNN
#             frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             img = Image.fromarray(frame_rgb)

#             # Detect and crop face
#             face_path = os.path.join(face_output_folder, f"face_{frame_id:04d}.jpg")
#             face = detector(img, save_path=face_path)

#             if face is not None:
#                 frame_id += 1
#             else:
#                 # Remove empty file if no face detected
#                 if os.path.exists(face_path):
#                     os.remove(face_path)

#         count += 1

#     cap.release()
#     return frame_id

# print("Face extraction function ready.")

In [ ]:
# # Clean previous output
# shutil.rmtree(FACE_OUTPUT, ignore_errors=True)
# os.makedirs(FACE_OUTPUT, exist_ok=True)

# # VIDEO_ROOT, FACE_OUTPUT, OUTPUT_DIR are already defined in config cell

# # Process all videos
# labels = ["Fake", "Real"]
# total_faces = 0

# for label in labels:
#     base_label_path = os.path.join(VIDEO_ROOT, label)

#     if not os.path.exists(base_label_path):
#         print(f"Warning: {base_label_path} not found, skipping...")
#         continue

#     if label == "Real":
#         processed_videos_count = 0
#         max_videos_for_label = MAX_REAL_VIDEOS
#         start_video_num = START_REAL_VIDEO_ID # Use the configured starting ID

#         print(f"Processing up to {max_videos_for_label} videos for label '{label}' starting from video ID {start_video_num}...")

#         all_real_videos_in_folder = sorted(os.listdir(base_label_path))

#         for vid_filename in all_real_videos_in_folder:
#             video_name_base = os.path.splitext(vid_filename)[0]

#             # Attempt to convert video name to integer for comparison
#             try:
#                 current_video_id = int(video_name_base)
#             except ValueError:
#                 # If the video name is not a number, skip it
#                 print(f"  [{label}] Skipping non-numeric video name: {vid_filename}")
#                 continue

#             # Skip videos before the specified starting point
#             if current_video_id < start_video_num:
#                 continue

#             # Apply the limit for videos processed *after* the starting point
#             if processed_videos_count >= max_videos_for_label:
#                 print(f"  Reached limit of {max_videos_for_label} videos for label '{label}' (after starting from {start_video_num}), skipping remaining.")
#                 break

#             if not vid_filename.endswith((".mp4", ".avi", ".mov", ".mkv")):
#                 continue

#             video_path = os.path.join(base_label_path, vid_filename)
#             face_dir = os.path.join(FACE_OUTPUT, label, video_name_base)
#             n_faces = extract_faces_from_video(video_path, face_dir)
#             total_faces += n_faces
#             processed_videos_count += 1
#             print(f"  [{label}] {video_name_base}: {n_faces} faces extracted")
#     elif label == "Fake":
#         print(f"Processing up to {MAX_FAKE_VIDEOS_PER_CATEGORY} videos per fake sub-category...")
#         # Iterate through sub-categories within 'Fake' folder
#         for sub_category in sorted(os.listdir(base_label_path)):
#             sub_category_path = os.path.join(base_label_path, sub_category)
#             if not os.path.isdir(sub_category_path):
#                 continue # Skip files, only process directories

#             vid_count_sub = 0
#             max_videos_for_sub_category = MAX_FAKE_VIDEOS_PER_CATEGORY
#             print(f"  Processing fake sub-category '{sub_category}'...")

#             for vid in sorted(os.listdir(sub_category_path)):
#                 if vid_count_sub >= max_videos_for_sub_category:
#                     print(f"    Reached limit of {max_videos_for_sub_category} videos for fake sub-category '{sub_category}', skipping remaining.")
#                     break
#                 if not vid.endswith((".mp4", ".avi", ".mov", ".mkv")):
#                     continue
#                 video_path = os.path.join(sub_category_path, vid)
#                 video_name = os.path.splitext(vid)[0]
#                 # Face output directory: Faces/Fake/{sub_category}/{video_name}/
#                 face_dir = os.path.join(FACE_OUTPUT, label, sub_category, video_name)
#                 n_faces = extract_faces_from_video(video_path, face_dir)
#                 total_faces += n_faces
#                 vid_count_sub += 1
#                 print(f"  [{label}/{sub_category}] {video_name}: {n_faces} faces extracted")

# print(f"\nTotal faces extracted: {total_faces}")

# 4. Build Dataset (Face-Only)

In [ ]:
from datasets import Dataset, DatasetDict, Image as HFImage

FACE_OUTPUT = "/content/drive/MyDrive/FYP/dataset/Frames" # Corrected path
OUTPUT_DIR = "/content/drive/MyDrive/FYP/TrainedModel_ViT"

# Collect all face image paths with labels
samples = []

for label_name, label_id in LABEL_MAP.items():
    label_path = Path(FACE_OUTPUT) / label_name

    if not label_path.exists():
        print(f"Warning: {label_path} not found")
        continue

    # Handle 'Fake' sub-categories
    if label_name == "Fake":
        for sub_category_folder in sorted(label_path.iterdir()):
            if not sub_category_folder.is_dir():
                continue
            for video_folder in sorted(sub_category_folder.iterdir()):
                if not video_folder.is_dir():
                    continue
                video_id = f"{sub_category_folder.name}/{video_folder.name}" # Include sub-category in video_id
                for file in sorted(os.listdir(video_folder)):
                    if file.endswith(".jpg"):
                        samples.append({
                            "image": str(video_folder / file),
                            "labels": label_id,
                            "video_id": video_id,
                        })
    else: # For 'Real' and any other single-level labels
        for video_folder in sorted(label_path.iterdir()):
            if not video_folder.is_dir():
                continue

            video_id = video_folder.name

            for file in sorted(os.listdir(video_folder)):
                if file.endswith(".jpg"):
                    samples.append({
                        "image": str(video_folder / file),
                        "labels": label_id,
                        "video_id": video_id,
                    })

print(f"Total face samples collected: {len(samples)}")
print(f"  Real: {sum(1 for s in samples if s['labels'] == 0)}")
print(f"  Fake: {sum(1 for s in samples if s['labels'] == 1)}")

Total face samples collected: 60261
  Real: 24245
  Fake: 36016


In [ ]:
from datasets import Features, ClassLabel

# Create dataset
dataset = Dataset.from_list(samples)

# Define the features with ClassLabel for 'labels'
features = Features({
    "image": dataset.features["image"],
    "labels": ClassLabel(num_classes=len(LABEL_MAP), names=list(LABEL_MAP.keys())),
    "video_id": dataset.features["video_id"],
})

# Cast the dataset to include the ClassLabel for 'labels'
dataset = dataset.cast(features)

# Proper 3-way split: 70% train, 15% validation, 15% test
train_test = dataset.train_test_split(test_size=0.3, seed=SEED, stratify_by_column="labels")
val_test = train_test["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="labels")

ds = DatasetDict({
    "train": train_test["train"],
    "validation": val_test["train"],
    "test": val_test["test"],
})

# Cast image column (this was already correct)
ds = ds.cast_column("image", HFImage())

print(f"Train:      {len(ds['train'])} samples")
print(f"Validation: {len(ds['validation'])} samples")
print(f"Test:       {len(ds['test'])} samples")

Casting the dataset:   0%|          | 0/60261 [00:00<?, ? examples/s]

Train:      42182 samples
Validation: 9039 samples
Test:       9040 samples


# 5. Image Preprocessing

In [ ]:
from transformers import AutoImageProcessor # User specified AutoImageProcessor, typically AutoVideoProcessor is used for video models

# Load the image processor for the chosen model
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

# Get normalization parameters (FIXED: mean and std are separate!)
img_mean = processor.image_mean
img_std = processor.image_std
size = processor.size.get("height", processor.size.get("shortest_edge", 224))

print(f"Image size: {size}")
print(f"Mean: {img_mean}")
print(f"Std:  {img_std}")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Image size: 224
Mean: [0.5, 0.5, 0.5]
Std:  [0.5, 0.5, 0.5]


In [ ]:
from torchvision.transforms import (
    Resize, CenterCrop, RandomHorizontalFlip, RandomRotation,
    Compose, ToTensor, Normalize, ColorJitter, GaussianBlur,
    RandomErasing
)
from torchvision import transforms

normalize = Normalize(mean=img_mean, std=img_std)

# Training transforms - strong augmentation for deepfake robustness
train_transform = Compose([
    Resize(size),
    CenterCrop(size),
    RandomHorizontalFlip(p=0.5),
    RandomRotation(degrees=10),
    ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomApply([GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.3),
    ToTensor(),
    normalize,
    RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

# Validation/Test transforms - no augmentation
eval_transform = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    normalize,
])

print("Transforms defined:")
print(f"  Train: {len(train_transform.transforms)} operations (with augmentation)")
print(f"  Eval:  {len(eval_transform.transforms)} operations (clean)")

Transforms defined:
  Train: 9 operations (with augmentation)
  Eval:  4 operations (clean)


In [ ]:
# Apply transforms to datasets
def apply_train_transform(example):
    example["pixel_values"] = [
        train_transform(image.convert("RGB")) for image in example["image"]
    ]
    return example

def apply_eval_transform(example):
    example["pixel_values"] = [
        eval_transform(image.convert("RGB")) for image in example["image"]
    ]
    return example

ds["train"].set_transform(apply_train_transform)
ds["validation"].set_transform(apply_eval_transform)
ds["test"].set_transform(apply_eval_transform)

# Verify
sample = ds["train"][0]
print(f"Sample pixel_values shape: {sample['pixel_values'].shape}")
print(f"Sample label: {ID2LABEL[sample['labels']]}")

Sample pixel_values shape: torch.Size([3, 224, 224])
Sample label: Real


# 6. Data Collator

In [ ]:
def collate_fn(examples):
    """Collate function for the Trainer."""
    pixel_values = torch.stack([ex["pixel_values"] for ex in examples])
    labels = torch.tensor([ex["labels"] for ex in examples])
    return {"pixel_values": pixel_values, "labels": labels}

# Quick test
from torch.utils.data import DataLoader

test_loader = DataLoader(ds["train"], collate_fn=collate_fn, batch_size=4)
batch = next(iter(test_loader))
print(f"Batch pixel_values: {batch['pixel_values'].shape}")
print(f"Batch labels:       {batch['labels'].shape}")

Batch pixel_values: torch.Size([4, 3, 224, 224])
Batch labels:       torch.Size([4])


# 7. Model Setup

Using **Swin Transformer** (`microsoft/swin-base-patch4-window7-224-in22k`)

**Why Swin for Deepfake Detection?**
- Hierarchical feature maps capture both fine-grained artifacts and global inconsistencies
- Shifted window attention is more efficient than global self-attention in ViT
- Achieves 97-99% accuracy on FaceForensics++ and CelebDF benchmarks
- ICCV 2021 Best Paper (Liu et al.)

In [ ]:
from transformers import ViTForImageClassification

# Load pre-trained ViT with custom classification head (2 classes)
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Model: google/vit-base-patch16-224
Total parameters:     85,800,194
Trainable parameters: 85,800,194


# 8. Evaluation Metrics

In [ ]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 during training."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    prec = precision_metric.compute(predictions=predictions, references=labels, average="binary")
    rec = recall_metric.compute(predictions=predictions, references=labels, average="binary")
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")

    return {
        "accuracy": acc["accuracy"],
        "precision": prec["precision"],
        "recall": rec["recall"],
        "f1": f1["f1"],
    }

print("Metrics loaded: accuracy, precision, recall, F1")

Metrics loaded: accuracy, precision, recall, F1


# 9. Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training schedule
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",

    # Batch sizes
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Logging
    logging_strategy="steps",
    logging_steps=10,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",  # Change to "wandb" or "tensorboard" if needed

    # Performance
    fp16=torch.cuda.is_available(),  # Mixed precision for speed
    dataloader_num_workers=2,
    remove_unused_columns=False,

    # Reproducibility
    seed=SEED,
)

print("Training configuration:")
print(f"  Epochs:         {NUM_EPOCHS}")
print(f"  Learning rate:  {LEARNING_RATE}")
print(f"  Batch size:     {TRAIN_BATCH_SIZE}")
print(f"  Warmup ratio:   {WARMUP_RATIO}")
print(f"  LR scheduler:   cosine")
print(f"  Mixed precision: {training_args.fp16}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training configuration:
  Epochs:         50
  Learning rate:  2e-05
  Batch size:     16
  Warmup ratio:   0.1
  LR scheduler:   cosine
  Mixed precision: True


# 10. Start Training

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Train!
print("Starting training...")
train_result = trainer.train()

# Print training summary
print("\n" + "="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"Training loss:    {train_result.training_loss:.4f}")
print(f"Training runtime: {train_result.metrics['train_runtime']:.1f}s")

# 40 minutes --> 0.13 epoch on swin 2
# ViT 18% times faster than swin

Starting training...


Epoch,Training Loss,Validation Loss


# 11. Evaluation on Test Set

In [ ]:
# Evaluate on truly unseen test data
test_results = trainer.predict(ds["test"])
print("\nTest Set Results:")
print(f"  Loss:      {test_results.metrics['test_loss']:.4f}")
print(f"  Accuracy:  {test_results.metrics['test_accuracy']:.4f}")
print(f"  Precision: {test_results.metrics['test_precision']:.4f}")
print(f"  Recall:    {test_results.metrics['test_recall']:.4f}")
print(f"  F1 Score:  {test_results.metrics['test_f1']:.4f}")

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

y_true = test_results.label_ids
y_pred = test_results.predictions.argmax(axis=1)
y_probs = torch.softmax(torch.tensor(test_results.predictions), dim=1).numpy()

# Classification Report
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_true, y_pred, target_names=["Real", "Fake"]))

# AUC-ROC Score
auc = roc_auc_score(y_true, y_probs[:, 1])
print(f"AUC-ROC: {auc:.4f}")

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
disp.plot(ax=axes[0], cmap="Blues", values_format="d")
axes[0].set_title("Confusion Matrix", fontsize=14, fontweight="bold")

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_probs[:, 1])
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {auc:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate", fontsize=12)
axes[1].set_ylabel("True Positive Rate", fontsize=12)
axes[1].set_title("ROC Curve", fontsize=14, fontweight="bold")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Evaluation plots saved!")

# 12. Save Model & Processor

In [ ]:
# Save the best model and processor
save_path = os.path.join(OUTPUT_DIR, "best_model")
trainer.save_model(save_path)
processor.save_pretrained(save_path)
print(f"Model and processor saved to: {save_path}")

# 13. Inference Pipeline

Predict whether a video is Real or Fake using the trained model.

In [ ]:
from facenet_pytorch import MTCNN
import torch.nn.functional as F
from transformers import AutoImageProcessor, ViTForImageClassification # Changed model import

def load_model(model_path):
    """Load trained model and processor."""
    proc = AutoImageProcessor.from_pretrained(model_path)
    mdl = ViTForImageClassification.from_pretrained(model_path) # Changed model class
    mdl.to(device)
    mdl.eval()
    return proc, mdl

# Face detector for inference
mtcnn = MTCNN(
    image_size=224,
    margin=20,
    device=device
)

def extract_frames_for_inference(video_path, frame_skip=10):
    """Extract frames from a video for prediction."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_skip == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
        count += 1

    cap.release()
    return frames

def detect_face(image, mtcnn_model):
    """Detect and crop face from a single frame."""
    face = mtcnn_model(image)
    if face is None:
        return None
    face = face.permute(1, 2, 0).cpu().numpy()
    face = Image.fromarray((face * 255).astype("uint8"))
    return face

def predict_single_image(image, proc, mdl):
    """Predict Real/Fake for a single face image."""
    inputs = proc(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = mdl(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    confidence = probs[0, pred].item()
    return pred, confidence, probs.cpu().numpy()[0]

def predict_video(video_path, proc, mdl, frame_skip=10):
    """
    Predict whether a video is Real or Fake.
    Uses majority voting + average confidence across detected faces.
    """
    frames = extract_frames_for_inference(video_path, frame_skip)

    if len(frames) == 0:
        return {"verdict": "Error", "reason": "Could not read video"}

    predictions = []
    all_probs = []

    for frame in frames:
        face = detect_face(frame, mtcnn)
        if face is None:
            continue

        pred, conf, probs = predict_single_image(face, proc, mdl)
        predictions.append(pred)
        all_probs.append(probs)

    if len(predictions) == 0:
        return {"verdict": "Unknown", "reason": "No faces detected in video"}

    # Average probability across all frames
    avg_probs = np.mean(all_probs, axis=0)
    final_pred = np.argmax(avg_probs)
    confidence = avg_probs[final_pred]

    # Vote counts
    vote_counts = Counter(predictions)

    return {
        "verdict": ID2LABEL[final_pred],
        "confidence": f"{confidence:.2%}",
        "real_prob": f"{avg_probs[0]:.4f}",
        "fake_prob": f"{avg_probs[1]:.4f}",
        "frames_analyzed": len(predictions),
        "vote_real": vote_counts.get(0, 0),
        "vote_fake": vote_counts.get(1, 0),
    }

print("Inference pipeline ready!")

ModuleNotFoundError: No module named 'facenet_pytorch'

# 14. Test Prediction

In [ ]:
# Load the trained model
model_path = os.path.join(OUTPUT_DIR, "best_model")
inf_processor, inf_model = load_model(model_path)

# Test on videos
test_videos = [
    ("/content/drive/MyDrive/FYP/dataset/Real.mp4", "Expected: Real"),
    ("/content/drive/MyDrive/FYP/dataset/01_02__exit_phone_room__YVGY8LOK.mp4", "Expected: Fake"),
]

print("="*60)
print("VIDEO PREDICTION RESULTS")
print("="*60)

for video_path, expected in test_videos:
    if not os.path.exists(video_path):
        print(f"\n⚠ Video not found: {video_path}")
        continue

    result = predict_video(video_path, inf_processor, inf_model)

    print(f"\n📹 {os.path.basename(video_path)}")
    print(f"   {expected}")
    for key, value in result.items():
        print(f"   {key}: {value}")

# 15. Single Image Prediction (Bonus)

In [ ]:
def predict_image_file(image_path, proc, mdl):
    """Predict Real/Fake for a single image file."""
    img = Image.open(image_path).convert("RGB")

    # Try face detection first
    face = detect_face(img, mtcnn)

    if face is not None:
        pred, confidence, probs = predict_single_image(face, proc, mdl)
        source = "face crop"
    else:
        # Fallback: use full image if no face detected
        pred, confidence, probs = predict_single_image(img, proc, mdl)
        source = "full image (no face detected)"

    print(f"Image: {os.path.basename(image_path)}")
    print(f"  Verdict:    {ID2LABEL[pred]}")
    print(f"  Confidence: {confidence:.2%}")
    print(f"  Real prob:  {probs[0]:.4f}")
    print(f"  Fake prob:  {probs[1]:.4f}")
    print(f"  Source:     {source}")

    return ID2LABEL[pred], confidence

# Example usage:
# predict_image_file("/path/to/image.jpg", inf_processor, inf_model)